In [19]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/nayla/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/nayla/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/nayla/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/nayla/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, row_number, sum as spark_sum
from pyspark.sql.window import Window
import numpy as np
import pandas as pd

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_2505060075_Nayla_Rahmadani") \
    .getOrCreate()

print("SparkSession berhasil diinisialisasi!")

SparkSession berhasil diinisialisasi!


In [22]:
# Baca data transaksi dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/nayla/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)

# Tambahkan kolom pendapatan (unit_terjual * harga_satuan)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# Buat df_target dari dictionary data_target_cabang
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Tampilkan beberapa baris awal
print("=== DataFrame Transaksi ===")
df_transaksi.show(5)

print("=== DataFrame Target Cabang ===")
df_target.show()

=== DataFrame Transaksi ===
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

=== DataFrame Target Cabang ===


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [23]:
# 1. Agregasi total pendapatan per kota
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 2. Join dengan df_target
df_bagian_a = df_pendapatan_kota.join(df_target, on="kota", how="inner")

# 3. Hitung pencapaian_persen dan urutkan secara descending
df_bagian_a = df_bagian_a.withColumn(
    "pencapaian_persen", 
    spark_round((col("total_pendapatan") / col("target_bulanan")) * 100, 2)
).orderBy(col("pencapaian_persen").desc())

print("=== Bagian A: Perbandingan Pencapaian Target Per Cabang ===")
df_bagian_a.show()

=== Bagian A: Perbandingan Pencapaian Target Per Cabang ===


+----------+----------------+--------------+----------+-----------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen|
+----------+----------------+--------------+----------+-----------------+
| Purworejo|        45650000|      30000000|     Fitri|           152.17|
|      Solo|        33475000|      40000000|      Bayu|            83.69|
|Yogyakarta|        47275000|      60000000|      Joko|            78.79|
|  Magelang|        31650000|      45000000|      Rani|            70.33|
|  Semarang|        38175000|      55000000|      Sari|            69.41|
+----------+----------------+--------------+----------+-----------------+



In [24]:
# 1. Agregasi total pendapatan per kota dan kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# 2. Definisi Window Partition per kota diurutkan berdasarkan pendapatan tertinggi
window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# 3. Terapkan row_number() dan ambil top-1 saja
df_bagian_b = df_kategori_kota.withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .orderBy(col("total_pendapatan_kategori").desc())

print("=== Bagian B: Kategori Terlaris per Kota ===")
df_bagian_b.show()

=== Bagian B: Kategori Terlaris per Kota ===


[Stage 12:>                                                         (0 + 1) / 1]

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|Yogyakarta|             Fashion|                 13325000|
|  Semarang|        Rumah Tangga|                 11125000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|  Magelang|Kesehatan & Kecan...|                  7275000|
+----------+--------------------+-------------------------+



In [25]:
# Register Temporary Views
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri Spark SQL Murni
query_c = """
SELECT 
    t.kota,
    tg.pic_cabang,
    COUNT(t.order_id) AS jumlah_transaksi
FROM transaksi t
JOIN target tg ON t.kota = tg.kota
GROUP BY t.kota, tg.pic_cabang
ORDER BY jumlah_transaksi DESC
"""

df_bagian_c = spark.sql(query_c)

print("=== Bagian C: Jumlah Transaksi per Kota (Spark SQL) ===")
df_bagian_c.show()

=== Bagian C: Jumlah Transaksi per Kota (Spark SQL) ===


[Stage 17:=============================>                            (3 + 3) / 6]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+

